In [ ]:
import socket, threading, time, statistics

HOST = "127.0.0.1"

def make_tcp_listener(port):
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind((HOST, port))
    srv.listen()
    return srv

def start_in_background(func, *args):
    t = threading.Thread(target=func, args=args, daemon=True)
    t.start()
    time.sleep(0.3)
    return t

In [ ]:
def tcp_echo_server(port, n_clients=1):
    srv = make_tcp_listener(port)
    print(f"[сервер] жду клиентов на {HOST}:{port}")
    for _ in range(n_clients):
        conn, addr = srv.accept()
        print(f"[сервер] подключился клиент {addr}")
        with conn:
            while True:
                data = conn.recv(1024)
                if not data:
                    break
                conn.sendall(data)
        print(f"[сервер] клиент {addr} отключился")
    srv.close()

def tcp_client(port, messages):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.connect((HOST, port))
        print("[клиент] мой адрес и порт:", s.getsockname())
        for text in messages:
            s.sendall(text.encode("utf-8"))
            reply = s.recv(1024).decode("utf-8")
            print(f"[клиент] отправил {text!r}, получил {reply!r}")

t = start_in_background(tcp_echo_server, 5001, 1)
tcp_client(5001, ["Привет", "Лабораторная работа 7", "Пока"])
t.join(timeout=2)

[сервер] жду клиентов на 127.0.0.1:5001
[сервер] подключился клиент ('127.0.0.1', 55588)
[клиент] мой адрес и порт: ('127.0.0.1', 55588)
[клиент] отправил 'Привет', получил 'Привет'
[клиент] отправил 'Лабораторная работа 7', получил 'Лабораторная работа 7'
[клиент] отправил 'Пока', получил 'Пока'
[сервер] клиент ('127.0.0.1', 55588) отключился


In [ ]:
print(len("Привет".encode("utf-8")))

12


In [ ]:
tcp_client(5999, ["тест"])

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
def udp_echo_server(port, n_messages):
    srv = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind((HOST, port))
    print(f"[UDP-сервер] жду датаграммы на {HOST}:{port}")
    for _ in range(n_messages):
        data, addr = srv.recvfrom(65535)
        srv.sendto(data, addr)
    srv.close()

def udp_client(port, messages):
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        s.settimeout(1.0)
        for text in messages:
            s.sendto(text.encode("utf-8"), (HOST, port))
            try:
                data, addr = s.recvfrom(65535)
                reply = data.decode("utf-8")
                print(f"[UDP-клиент] отправил {text!r}, получил {reply!r} от {addr}")
            except (socket.timeout, ConnectionResetError,
                    ConnectionRefusedError) as e:
                print(f"[UDP-клиент] ответа нет: {type(e).__name__}")

t = start_in_background(udp_echo_server, 5002, 3)
udp_client(5002, ["один", "два", "три"])

[UDP-сервер] жду датаграммы на 127.0.0.1:5002
[UDP-клиент] отправил 'один', получил 'один' от ('127.0.0.1', 5002)
[UDP-клиент] отправил 'два', получил 'два' от ('127.0.0.1', 5002)
[UDP-клиент] отправил 'три', получил 'три' от ('127.0.0.1', 5002)


In [ ]:
udp_client(5999, ["тест"])

[UDP-клиент] ответа нет: TimeoutError


In [ ]:
t = start_in_background(udp_echo_server, 5002, 1)
udp_client(5002, ["тест"])

[UDP-сервер] жду датаграммы на 127.0.0.1:5002
[UDP-клиент] отправил 'тест', получил 'тест' от ('127.0.0.1', 5002)


In [ ]:
def slow_handler(conn):
    with conn:
        data = conn.recv(1024)
        time.sleep(1.0)                       # имитация долгой работы
        conn.sendall(data.upper())

def sequential_server(port, n_clients):
    srv = make_tcp_listener(port)
    for _ in range(n_clients):
        conn, addr = srv.accept()
        slow_handler(conn)                    # остальные клиенты ждут
    srv.close()

def threaded_server(port, n_clients):
    srv = make_tcp_listener(port)
    workers = []
    for _ in range(n_clients):
        conn, addr = srv.accept()
        w = threading.Thread(target=slow_handler, args=(conn,))
        w.start()                             # каждый клиент - в своём потоке
        workers.append(w)
    for w in workers:
        w.join()
    srv.close()

def one_client(port, name, results):
    t0 = time.perf_counter()
    with socket.create_connection((HOST, port)) as s:
        s.sendall(name.encode("utf-8"))
        s.recv(1024)
    results[name] = time.perf_counter() - t0

def run_clients(port, n):
    results = {}
    threads = [threading.Thread(target=one_client,
                                args=(port, f"client{i}", results))
               for i in range(1, n + 1)]
    t0 = time.perf_counter()
    for th in threads:
        th.start()
    for th in threads:
        th.join()
    total = time.perf_counter() - t0
    for name in sorted(results):
        print(f"  {name}: ответ через {results[name]:.2f} с")
    print(f"  Всего: {total:.2f} с")

print("Последовательный сервер:")
start_in_background(sequential_server, 5003, 5)
run_clients(5003, 5)

print("Многопоточный сервер:")
start_in_background(threaded_server, 5004, 5)
run_clients(5004, 5)

Последовательный сервер:
  client1: ответ через 3.00 с
  client2: ответ через 1.00 с
  client3: ответ через 2.00 с
  client4: ответ через 4.00 с
  client5: ответ через 5.00 с
  Всего: 5.00 с
Многопоточный сервер:
  client1: ответ через 1.00 с
  client2: ответ через 1.00 с
  client3: ответ через 1.00 с
  client4: ответ через 1.00 с
  client5: ответ через 1.00 с
  Всего: 1.00 с


In [ ]:
def recv_exact(sock, n):
    buf = b""
    while len(buf) < n:
        part = sock.recv(n - len(buf))
        if not part:
            raise ConnectionError("соединение закрыто")
        buf += part
    return buf

def tcp_fixed_echo_server(port, size):
    srv = make_tcp_listener(port)
    conn, _ = srv.accept()
    with conn:
        while True:
            try:
                data = recv_exact(conn, size)
            except ConnectionError:
                break
            conn.sendall(data)
    srv.close()

def measure_tcp_rtt(port, size, n):
    start_in_background(tcp_fixed_echo_server, port, size)
    msg = b"x" * size
    rtts = []
    with socket.create_connection((HOST, port)) as s:
        for _ in range(n):
            t0 = time.perf_counter()
            s.sendall(msg)
            recv_exact(s, size)
            rtts.append((time.perf_counter() - t0) * 1000)   # миллисекунды
    return rtts

def measure_udp_rtt(port, size, n):
    start_in_background(udp_echo_server, port, n)
    msg = b"x" * size
    rtts, lost = [], 0
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        s.settimeout(1.0)
        for _ in range(n):
            t0 = time.perf_counter()
            s.sendto(msg, (HOST, port))
            try:
                s.recvfrom(65535)
                rtts.append((time.perf_counter() - t0) * 1000)
            except socket.timeout:
                lost += 1
    return rtts, lost

def report(name, rtts):
    print(f"{name}: среднее {statistics.mean(rtts):.3f} мс, "
          f"медиана {statistics.median(rtts):.3f} мс, "
          f"мин {min(rtts):.3f} мс, макс {max(rtts):.3f} мс")

SIZE, N = 64, 1000
report("TCP", measure_tcp_rtt(5005, SIZE, N))
udp_rtts, lost = measure_udp_rtt(5006, SIZE, N)
report("UDP", udp_rtts)
print("Потеряно UDP-датаграмм:", lost)

TCP: среднее 0.034 мс, медиана 0.033 мс, мин 0.021 мс, макс 0.143 мс
[UDP-сервер] жду датаграммы на 127.0.0.1:5006
UDP: среднее 0.041 мс, медиана 0.039 мс, мин 0.021 мс, макс 0.252 мс
Потеряно UDP-датаграмм: 0


In [ ]:
def sink_server(port):
    srv = make_tcp_listener(port)
    conn, _ = srv.accept()
    total = 0
    with conn:
        while True:
            data = conn.recv(65536)
            if not data:                      # клиент закончил передачу
                break
            total += len(data)
        conn.sendall(str(total).encode())     # отвечаем числом принятых байт
    srv.close()

def measure_throughput(port, total_mb, chunk):
    start_in_background(sink_server, port)
    block = b"x" * chunk
    n_blocks = total_mb * 1024 * 1024 // chunk
    with socket.create_connection((HOST, port)) as s:
        t0 = time.perf_counter()
        for _ in range(n_blocks):
            s.sendall(block)
        s.shutdown(socket.SHUT_WR)            # "я всё отправил", но жду ответ
        received = int(s.recv(64).decode())
        dt = time.perf_counter() - t0
    mbit_s = received * 8 / dt / 1_000_000
    print(f"блок {chunk:>6} Б: принято {received} Б за {dt:.3f} с"
          f" -> {mbit_s:.0f} Мбит/с")
    return mbit_s

for i, chunk in enumerate([64, 512, 4096, 65536]):
    measure_throughput(5010 + i, 20, chunk)

блок     64 Б: принято 20971520 Б за 1.494 с -> 112 Мбит/с
блок    512 Б: принято 20971520 Б за 0.064 с -> 2637 Мбит/с
блок   4096 Б: принято 20971520 Б за 0.019 с -> 8804 Мбит/с
блок  65536 Б: принято 20971520 Б за 0.006 с -> 28770 Мбит/с
